In [1]:
import pandas as pd
import nfl_data_py as nfl

In [7]:
from data.data_aggregation import import_pbp_data

In [45]:
DATA_DIR = r"C://Users//natel//PycharmProjects//NFL_Overtime_Model//data"
df = pd.read_pickle(f'{DATA_DIR}//pbp_data.pkl')
df['game_id'].min()

'2006_01_ATL_CAR'

In [47]:
import pandas as pd

def get_ruleset(season, week):
    is_playoff = week > 18
    if season >= 2025:
        return 'post2025'
    elif season >= 2022 and is_playoff:
        return 'post2025'  # same rules as post2025
    elif season >= 2012:
        return '2012_2024'
    else:
        return 'pre2012'

def aggregate_overtime_games(pbp: pd.DataFrame) -> pd.DataFrame:
    ot_plays = pbp[pbp['game_half'] == 'Overtime'].copy()

    results = []

    for game_id, game in ot_plays.groupby('game_id'):
        game = game.sort_values('play_id')

        first_play = game.iloc[0]
        season = first_play['season']
        week = first_play['week']
        receiving_team = first_play['posteam']

        home = first_play['home_team']
        away = first_play['away_team']
        kicking_team = away if receiving_team == home else home

        last_play = game.iloc[-1]
        home_score = last_play['total_home_score']
        away_score = last_play['total_away_score']

        if home_score > away_score:
            winner = home
        elif away_score > home_score:
            winner = away
        else:
            winner = 'TIE'

        results.append({
            'game_id': game_id,
            'season': season,
            'week': week,
            'ruleset': get_ruleset(season, week),
            'kicking_team': kicking_team,
            'receiving_team': receiving_team,
            'winner': winner,
            'kicking_team_won': winner == kicking_team,
            'receiving_team_won': winner == receiving_team,
            'tie': winner == 'TIE',
        })

    return pd.DataFrame(results).sort_values(['season', 'week']).reset_index(drop=True)

In [48]:
ot_df = aggregate_overtime_games(df)

In [49]:
ot_df.groupby('ruleset').agg(
    games=('game_id', 'count'),
    kicking_won=('kicking_team_won', 'sum'),
    receiving_won=('receiving_team_won', 'sum'),
    ties=('tie', 'sum'),
    kicking_pct=('kicking_team_won', 'mean'),
    receiving_pct=('receiving_team_won', 'mean'),
    tie_pct=('tie', 'mean'),
).round(3).T


ruleset,2012_2024,post2025,pre2012
games,211.000,17.000,93.000
kicking_won,83.000,7.000,43.000
receiving_won,113.000,9.000,49.000
ties,15.000,1.000,1.000
kicking_pct,0.393,0.412,0.462
receiving_pct,0.536,0.529,0.527
tie_pct,0.071,0.059,0.011
